# ML Training Pipeline - Notebook de Recherche

Ce notebook demontre le workflow complet du pipeline ML :
1. Chargement des donnees OHLCV
2. Visualisation et exploration
3. Feature engineering
4. Entrainement des modeles (dry-run)
5. Evaluation des checkpoints sauvegardes

**Pour un entrainement GPU complet**, utiliser les scripts `train_*.py` en CLI.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Dependances chargees")
print(f"Working dir: {Path.cwd()}")

## 1. Chargement des Donnees

Utilise les donnees telechargees via `scripts/datasets/` ou des donnees synthetiques pour demo.

In [ ]:
from train_classification import generate_synthetic_data, load_data, generate_features

data_dir = Path("../datasets/yfinance")
symbol = "SPY"

if data_dir.exists():
    raw = load_data(data_dir, symbol)
    print(f"Donnees reelles : {len(raw)} lignes, {raw.index.min().date()} -> {raw.index.max().date()}")
else:
    raw = generate_synthetic_data(3000)
    print(f"Donnees synthetiques : {len(raw)} lignes")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(raw.index, raw["Close"], linewidth=0.8)
axes[0].set_title(f"{symbol} - Prix de cloture")
axes[0].set_ylabel("Prix")

returns = raw["Close"].pct_change().dropna()
axes[1].bar(returns.index, returns.values, width=1, alpha=0.7)
axes[1].set_title("Rendements quotidiens")
axes[1].set_ylabel("Rendement")

plt.tight_layout()
plt.show()

print(f"Rendement moyen: {returns.mean():.5f}")
print(f"Volatilite annualisee: {returns.std() * np.sqrt(252):.4f}")

## 2. Feature Engineering

Creation des features techniques : rendements multi-horizon, volatilite, RSI, MACD, Bollinger Bands.

In [ ]:
features = generate_features(raw, lookback=20)
print(f"Features generees : {features.shape}")
print(f"Colonnes : {list(features.columns)}")

features.describe()

In [ ]:
target_col = "target"
if target_col in features.columns:
    corr = features.corr()[target_col].drop(target_col).sort_values()

    fig, ax = plt.subplots(figsize=(10, 6))
    corr.plot(kind="barh", ax=ax)
    ax.set_title("Correlation avec la cible")
    ax.axvline(0, color="black", linewidth=0.5)
    plt.tight_layout()
    plt.show()

## 3. Entrainement - Dry Run

Validation du pipeline avec donnees synthetiques. Pour un entrainement complet sur GPU, utiliser :
```bash
python scripts/train_lstm.py --data-dir ../datasets/yfinance --symbol SPY --hidden-size 512 --epochs 200
```

In [ ]:
from train_classification import train_and_evaluate, compute_data_hash

synth = generate_synthetic_data(500)
synth_features = generate_features(synth)
data_hash = compute_data_hash(synth)

result_rf = train_and_evaluate(synth_features, model_type="rf", n_estimators=50, max_depth=5)
print(f"RandomForest : Accuracy={result_rf['metrics']['accuracy']}, F1={result_rf['metrics']['f1']}")

In [ ]:
from train_lstm import build_sequences, normalize_sequences
from train_lstm import train_and_evaluate as train_lstm_model

lstm_features = generate_features(synth)
X, y, feat_cols = build_sequences(lstm_features, seq_len=20)

split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
X_train, X_test, _, _ = normalize_sequences(X_train, X_test)

lstm_result = train_lstm_model(X_train, y_train, X_test, y_test,
    hidden_size=64, num_layers=1, epochs=2, batch_size=32, device="cpu")
print(f"LSTM : MSE={lstm_result['metrics']['mse']:.6f}, DirAcc={lstm_result['metrics']['direction_accuracy']}")

## 4. Evaluation d'un Checkpoint

Chargement et evaluation d'un modele sauvegarde. Requiert un checkpoint existant.

In [ ]:
import json

ckpt_base = Path("../checkpoints")
all_ckpts = []

if ckpt_base.exists():
    for model_dir in ckpt_base.iterdir():
        if model_dir.is_dir():
            for ts_dir in model_dir.iterdir():
                if ts_dir.is_dir() and (ts_dir / "metadata.json").exists():
                    meta = json.loads((ts_dir / "metadata.json").read_text())
                    all_ckpts.append({"path": str(ts_dir), **meta})

if all_ckpts:
    latest = sorted(all_ckpts, key=lambda x: x["timestamp"])[-1]
    print(f"Checkpoint le plus recent : {latest['path']}")
    print(f"Modele : {latest['model_type']}")
    print(f"Metriques : {latest['metrics']}")
    print(f"Data hash : {latest['data_hash']}")
else:
    print("Aucun checkpoint trouve. Lancez un entrainement d'abord.")

## 5. Resume

| Etape | Status |
|-------|--------|
| Chargement donnees | OK |
| Feature engineering | OK |
| Dry-run classification | OK |
| Dry-run LSTM | OK |
| Checkpoint evaluation | A verifier apres entrainement |

Prochaines etapes : lancer les entrainements GPU complets via les scripts CLI.